# Python Q&A Assistant — API Test Results

Tests 10 diverse Python queries against the deployed API.
Covers beginner, intermediate, advanced, edge cases, and error handling.

In [ ]:
import requests
import json
import time
import pandas as pd

BASE_URL = "http://localhost:8000"  # change to deployed URL when live

def ask(question: str) -> dict:
    start = time.time()
    resp = requests.post(f"{BASE_URL}/ask", json={"question": question}, timeout=120)
    resp.raise_for_status()
    data = resp.json()
    data["_request_latency_ms"] = round((time.time() - start) * 1000, 1)
    return data

print("API ready:", requests.get(f"{BASE_URL}/health").json())

## Test 1 — Beginner: List Reversal

In [ ]:
r1 = ask("How do I reverse a list in Python?")
print("Q:", r1["question"])
print("Rewritten:", r1["rewritten_query"])
print("Answer:\n", r1["answer"])
print("Latency:", r1["latency_ms"], "ms")
print("Hallucination detected:", r1["hallucination_detected"])
print("Sources:", [s['title'] for s in r1['sources']])

## Test 2 — Intermediate: List Comprehension vs Generator

In [ ]:
r2 = ask("What is the difference between list comprehension and generator expression in Python?")
print("Answer:\n", r2["answer"])
print("Latency:", r2["latency_ms"], "ms | Cache hit:", r2["cache_hit"])

## Test 3 — Intermediate: Decorators

In [ ]:
r3 = ask("How do Python decorators work? Can you show an example?")
print("Answer:\n", r3["answer"])
print("Retry count:", r3["retry_count"])

## Test 4 — Intermediate: @staticmethod vs @classmethod

In [ ]:
r4 = ask("What is the difference between @staticmethod and @classmethod in Python?")
print("Answer:\n", r4["answer"])

## Test 5 — Advanced: GIL and Threading

In [ ]:
r5 = ask("What is the Python GIL and how does it affect multithreading performance?")
print("Answer:\n", r5["answer"])
print("Sources:", [s['title'] for s in r5['sources']])

## Test 6 — Advanced: Async/Await

In [ ]:
r6 = ask("How do async and await work in Python? When should I use asyncio?")
print("Answer:\n", r6["answer"])

## Test 7 — Library: Pandas GroupBy

In [ ]:
r7 = ask("How do I use groupby in pandas to aggregate data?")
print("Answer:\n", r7["answer"])

## Test 8 — Edge Case: Empty/Vague Question

In [ ]:
try:
    r8 = ask("Python")
    print("Answer:\n", r8["answer"])
    print("Note: vague query handled gracefully")
except Exception as e:
    print("Edge case handled with error:", str(e))

## Test 9 — Semantic Cache: Repeat Similar Question

In [ ]:
# Ask similar question to Test 1 — should hit cache
r9 = ask("What's the best way to reverse a Python list?")
print("Cache hit:", r9["cache_hit"])
print("Cache similarity:", r9.get("cache_similarity"))
print("Latency (cached):", r9["latency_ms"], "ms")

## Test 10 — Advanced: Memory Management / __slots__

In [ ]:
r10 = ask("What are __slots__ in Python and how do they reduce memory usage?")
print("Answer:\n", r10["answer"])
print("Answer grade:", r10["answer_grade"])

## Summary Table

In [ ]:
results = [
    {"#": 1, "Question": "How to reverse a list?", "Latency (ms)": r1["latency_ms"], "Cache Hit": r1["cache_hit"], "Hallucination": r1["hallucination_detected"], "Grade": r1["answer_grade"]},
    {"#": 2, "Question": "List comp vs generator?", "Latency (ms)": r2["latency_ms"], "Cache Hit": r2["cache_hit"], "Hallucination": r2["hallucination_detected"], "Grade": r2["answer_grade"]},
    {"#": 3, "Question": "How do decorators work?", "Latency (ms)": r3["latency_ms"], "Cache Hit": r3["cache_hit"], "Hallucination": r3["hallucination_detected"], "Grade": r3["answer_grade"]},
    {"#": 4, "Question": "staticmethod vs classmethod?", "Latency (ms)": r4["latency_ms"], "Cache Hit": r4["cache_hit"], "Hallucination": r4["hallucination_detected"], "Grade": r4["answer_grade"]},
    {"#": 5, "Question": "Python GIL and threading?", "Latency (ms)": r5["latency_ms"], "Cache Hit": r5["cache_hit"], "Hallucination": r5["hallucination_detected"], "Grade": r5["answer_grade"]},
    {"#": 6, "Question": "How does async/await work?", "Latency (ms)": r6["latency_ms"], "Cache Hit": r6["cache_hit"], "Hallucination": r6["hallucination_detected"], "Grade": r6["answer_grade"]},
    {"#": 7, "Question": "Pandas groupby aggregate?", "Latency (ms)": r7["latency_ms"], "Cache Hit": r7["cache_hit"], "Hallucination": r7["hallucination_detected"], "Grade": r7["answer_grade"]},
    {"#": 8, "Question": "Python (edge case)", "Latency (ms)": 0, "Cache Hit": False, "Hallucination": False, "Grade": "edge_case"},
    {"#": 9, "Question": "Reverse a Python list? (cache)", "Latency (ms)": r9["latency_ms"], "Cache Hit": r9["cache_hit"], "Hallucination": r9["hallucination_detected"], "Grade": r9["answer_grade"]},
    {"#": 10, "Question": "__slots__ memory?", "Latency (ms)": r10["latency_ms"], "Cache Hit": r10["cache_hit"], "Hallucination": r10["hallucination_detected"], "Grade": r10["answer_grade"]},
]

df = pd.DataFrame(results)
print(df.to_string(index=False))
print(f"\nAvg latency (non-cached): {df[~df['Cache Hit']]['Latency (ms)'].mean():.0f} ms")
print(f"Cache hit rate: {df['Cache Hit'].mean():.0%}")